In [1]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import ast
import pandas as pd

In [2]:
# Paths
base_dir = Path("../..")
# json_path = "groups_same_first.json"
csv_path = "groups_same_first_global_results.csv"
img_dir = base_dir / "data" / "floorplan_image"
txt_dir = base_dir / "annotation" / "human_annotated_tags"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Load JSON entries
# with open(json_path, 'r') as f:
#     entries = json.load(f)

# Prepare font for subtitles
try:
    font = ImageFont.truetype("arial.ttf", size=16)
except IOError:
    font = ImageFont.load_default()

subtitle_height = 20  # space for ID subtitles

df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

for idx, entry in enumerate(entries, start=1):
    group_ids = entry
    images = []

    # Load images for this group
    for img_id in group_ids:
        img_path = img_dir / f"{img_id}.png"
        if img_path.exists():
            images.append((img_id, Image.open(img_path)))
        else:
            raise FileNotFoundError(f"Image file not found: {img_path}")

    # Determine layout
    widths, heights = zip(*(im.size for _, im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new('RGB', (total_width, max_height + subtitle_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # Paste images and draw subtitles
    x_offset = 0
    for img_id, im in images:
        canvas.paste(im, (x_offset, 0))
        # Compute text size
        if hasattr(font, 'getsize'):
            text_w, text_h = font.getsize(img_id)
        else:
            bbox = draw.textbbox((0,0), img_id, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        text_x = x_offset + (im.width - text_w) // 2
        text_y = max_height + (subtitle_height - text_h) // 2
        draw.text((text_x, text_y), img_id, fill=(0, 0, 0), font=font)
        x_offset += im.width

    # Save combined image
    img_out_path = out_dir / f"example_{idx}_2.png"
    canvas.save(img_out_path)

    # # Collect and write text descriptions
    # txt_out_path = out_dir / f"example_{idx}.txt"
    # with open(txt_out_path, 'w') as txt_out:
    #     for img_id, _ in images:
    #         tag_file = txt_dir / f"{img_id}.txt"
    #         if tag_file.exists():
    #             with open(tag_file, 'r') as tf:
    #                 desc = tf.read().strip()
    #         else:
    #             raise FileNotFoundError(f"Tag file not found: {tag_file}")
    #         txt_out.write(f"ID: {img_id}, description: {desc}\n\n")

print(f"Generated {len(entries)} example image/text pairs in '{out_dir.resolve()}'")



KeyError: 'options'

In [3]:
import glob

In [36]:
input_folder  = 'examples_same_second'
output_folder = os.path.join(input_folder, 'combined')
os.makedirs(output_folder, exist_ok=True)

# Find every file that ends with '_2.png'
pattern = os.path.join(input_folder, '*_2.png')
for suffix_path in glob.glob(pattern):
    # Derive the base filename by stripping off '_2' before the extension
    folder, suffix_fn = os.path.split(suffix_path)
    stem = suffix_fn[:-6]  # removes the trailing "_2.png"
    base_fn = stem + '.png'
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"No base image found for {suffix_fn}, skipping.")
        continue

    # Open both images (they must be same size)
    img_base   = Image.open(base_path)
    img_suffix = Image.open(suffix_path)
    w, h       = img_base.size

    # Stack base on top of suffix
    combined = Image.new('RGB', (w, 2*h))
    combined.paste(img_base,   (0, 0))
    combined.paste(img_suffix, (0, h))

    # Save
    out_fn = f"{stem}_combined.png"
    combined.save(os.path.join(output_folder, out_fn))



No base image found for example_2.png, skipping.
